# **Investment Portfolio Management**

## **About the scenario**
This scenario demonstrates best practices in a data engineering context, where notebooks are used to orchestrate complex tasks involving real-time data fetching, computation, and report delivery. The notebook is modular, and secure, employing batch processing, error handling, and proper separation of concerns.

In this scenario, we will:

1. Ingest a CSV file containing the user’s investment portfolio.
2. Fetch real-time stock prices via an API using *Function Calling*.
3. Perform calculations on the portfolio, including total value and percentage changes, using *Code Interpreter*.
4. Generate a detailed report summarizing the portfolio's performance.
5. Save the report to Azure Blob Storage using *Function Calling*.

## **Time**
You should expect to spend 10-15 minutes building and running this scenario. 

## **Before you begin**

### Install required libraries

In [ ]:
# Install the packages
%pip install -r ./requirements.txt

### Set parameters

### Setup credentials

## **Ingest data**
This scenario ingests data from two separte sources:
- Files from the folder [`data/`](./data/) in this repo. This file simulates a personal investment portfolio with ticker `Symbol`, `Average_Cost`, `Quantity` data. You can clone this repo or copy this folder to make sure you have access to these files when running this scenario.
- Latest stock prices by ticker symbol using Yahoo Finance ([`yfinance`](https://pypi.org/project/yfinance/)).

### Personal Investment Portfolio Data

-- bigger better dataset with wget and get table. 

In [66]:
import pandas as pd

# Load the investment portfolio CSV file
portfolio_df = pd.read_csv('./data/portfolio.csv')
portfolio_df.head()

,Symbol,Average_Cost,QTY
0,MSFT,200,300
1,AAPL,114,200
2,amzn,125,50
3,TSLA,900,100
4,NfLx,540,80


### Get Latest Stock Prices
This function retrieves the stock data for a specified `ticker` symbol using the yfinance library, specifically pulling the latest data for the last trading day.

In [70]:
import yfinance as yf

def fetch_stock_price(ticker_symbol: str) -> dict:
    """
    Fetch the latest stock price and opening price for a given ticker symbol.

    Parameters:
    - ticker_symbol (str): The ticker symbol of the stock to retrieve data for.

    Returns:
    - dict: A dictionary containing the following keys:
        - "company_name" (str): The name of the company corresponding to the ticker symbol.
        - "ticker" (str): The ticker symbol provided as input.
        - "open_price" (float): The opening price of the stock for the latest trading day.
        - "latest_close_price" (float): The closing price of the stock for the latest trading day.
        - "error" (str): An error message if an issue occurred, or if no data was found.
        
    Example:
    >>> fetch_stock_price("AAPL")
    {'ticker': 'AAPL', 'open_price': 145.3, 'latest_close_price': 148.9}
    """

    try:
        # Fetch the stock's trading history for the last day
        stock = yf.Ticker(ticker_symbol)
        stock_data = stock.history(period="1d")

        # Check if the data is empty, indicating an invalid ticker or no data available
        if stock_data.empty:
            return {"error": f"No data found for ticker symbol: {ticker_symbol}"}

         # Extract the company name from the 'info' dictionary
        company_name = stock.info.get('longName', "Company name not available")

        # Extract the latest available open and close prices
        latest_open_price = stock_data['Open'].iloc[-1]
        latest_close_price = stock_data['Close'].iloc[-1]
        
        # TODO: logging
        #print(f"{company_name}(Ticker): {ticker_symbol} | Open Price: {round(latest_open_price, 2)} | Close Price: {round(latest_close_price, 2)}")

        return {
            "company_name": company_name,
            "ticker": ticker_symbol,
            "latest_open_price": latest_open_price,
            "latest_close_price": latest_close_price
        }

    except KeyError as key_error:
        return {"error": f"Data missing for key: {key_error}. Check if the ticker is correct."}

    except yf.YFinanceError as yf_error:
        return {"error": f"yfinance error occurred: {yf_error}"}

    except Exception as generic_error:
        return {"error": f"An unexpected error occurred: {generic_error}"}


In [71]:
###################
#### DEBUGGING ####
###################

## Fetch the stock price for each stock in the portfolio
stock_prices = []
for ticker in portfolio_df['Symbol']:
    stock_price = fetch_stock_price(ticker)
    stock_prices.append(stock_price)

# Create a dataframe from the stock prices list
stock_prices_df = pd.DataFrame(stock_prices)
display(stock_prices_df)

,company_name,ticker,latest_open_price,latest_close_price
0,Microsoft Corporation,MSFT,431.654999,427.619995
1,Apple Inc.,AAPL,233.315002,234.449997
2,"Amazon.com, Inc.",amzn,189.580002,189.300003
3,"Tesla, Inc.",TSLA,269.877014,269.130005
4,"Netflix, Inc.",NfLx,758.679993,751.080017
5,NVIDIA Corporation,NVDA,143.029999,140.941101


## **Transformation**

--- add some error ticker
-- normalize

In [68]:
# Define function to cleanse and format the stock prices dataframe
def cleanse_stock_prices(stock_prices_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleanse and format the stock prices dataframe.

    Parameters:
    - stock_prices_df (pd.DataFrame): The dataframe containing stock prices data.

    Returns:
    - pd.DataFrame: A new dataframe containing the cleansed and formatted data.
    """
    try:
        # Remove rows with missing or non-positive prices
        stock_prices_df = stock_prices_df.dropna()
        stock_prices_df = stock_prices_df[stock_prices_df['latest_close_price'] > 0]

        # Ensure the company name is in title case
        stock_prices_df['company_name'] = stock_prices_df['company_name'].apply(lambda x: x.title() if x else "Company name not available")

        # Ensure the ticker symbol is in uppercase
        stock_prices_df['ticker'] = stock_prices_df['ticker'].str.upper()

        # Round the prices to 2 decimal places
        stock_prices_df['latest_open_price'] = stock_prices_df['latest_open_price'].round(2)  
        stock_prices_df['latest_close_price'] = stock_prices_df['latest_close_price'].round(2)

        # Reset the index of the dataframe after removing rows and columns to ensure it is continuous
        stock_prices_df = stock_prices_df.reset_index(drop=True)

        return stock_prices_df

    except KeyError as key_error:
        print(f"KeyError: {key_error}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error


In [72]:
###################
#### DEBUGGING ####
###################

print(f"Type of stock_prices_df: {type(stock_prices_df)}")

display(stock_prices_df)

# Cleanse the stock prices dataframe
clean_stock_prices_df = cleanse_stock_prices(stock_prices_df)
display(clean_stock_prices_df)

Type of stock_prices_df: <class 'pandas.core.frame.DataFrame'>


,company_name,ticker,latest_open_price,latest_close_price
0,Microsoft Corporation,MSFT,431.654999,427.619995
1,Apple Inc.,AAPL,233.315002,234.449997
2,"Amazon.com, Inc.",amzn,189.580002,189.300003
3,"Tesla, Inc.",TSLA,269.877014,269.130005
4,"Netflix, Inc.",NfLx,758.679993,751.080017
5,NVIDIA Corporation,NVDA,143.029999,140.941101


,company_name,ticker,latest_open_price,latest_close_price
0,Microsoft Corporation,MSFT,431.65,427.62
1,Apple Inc.,AAPL,233.32,234.45
2,"Amazon.Com, Inc.",AMZN,189.58,189.30
3,"Tesla, Inc.",TSLA,269.88,269.13
4,"Netflix, Inc.",NFLX,758.68,751.08
5,Nvidia Corporation,NVDA,143.03,140.94


## **Analyze**

- make sure the datasets match, names, tickers, values formatted the same
- upload the two files (export df CSV files) file search API
- assistant with files mounted -- code inter
    - assistnent with code interp and files mounted.
    - ask questions